# Bab 10 - Neural Learning about Edges and Corners: Convolutional Neural Networks

**Nama:** Nanda Pratama ITM  
**NIM:** 101032330212  
**Kelas:** TK 47 01


> Notebook ini merupakan reproduksi edukatif dan pendalaman teori berdasarkan *Grokking Deep Learning* (Andrew W. Trask, Manning, 2019). Kode inti diadaptasi dari buku dan repositori pendamping resmi; perubahan kompatibilitas dijelaskan dalam dokumentasi.


## Capaian pembelajaran

Setelah menyelesaikan notebook ini, pembaca diharapkan mampu menjelaskan konsep utama bab, menghubungkannya dengan operasi NumPy, menjalankan contoh secara berurutan, dan menyebutkan batas interpretasinya.


## Local connectivity

Citra mempunyai struktur lokal: piksel berdekatan lebih berkaitan daripada piksel jauh. Convolutional layer memproses patch kecil sehingga parameter fokus pada pola lokal seperti tepi dan sudut.


## Weight sharing

Kernel yang sama digeser ke banyak lokasi. Berbagi bobot mengurangi jumlah parameter dan memberi sensitivitas terhadap pola yang muncul di lokasi berbeda.


## Bentuk keluaran

Tanpa padding, keluaran satu dimensi berukuran $\lfloor (N-K)/S\rfloor+1$, dengan ukuran input $N$, kernel $K$, dan stride $S$. Banyak kernel menghasilkan banyak feature map.


## Belajar kernel

Kernel bukan harus filter buatan tangan; gradien dari loss dapat memperbaruinya. Lapisan berikut menggabungkan respons lokal untuk klasifikasi. Translational invariance biasanya diperkuat dengan pooling atau augmentasi.


## Reproduksi kode dan eksperimen

Jalankan sel dari atas ke bawah. `MODE_CEPAT=True` menjaga waktu eksekusi singkat; ubah ke `False` untuk konfigurasi yang lebih panjang.


In [1]:
from pathlib import Path
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(1)
DATA_DIR = (Path.cwd().parent / "data").resolve()
MODE_CEPAT = True  # ubah ke False untuk eksperimen lebih panjang
assert DATA_DIR.is_dir(), f"Folder data tidak ditemukan: {DATA_DIR}"
print("Data:", DATA_DIR)
print("Mode cepat:", MODE_CEPAT)

def sigmoid(x):
    x = np.clip(x, -60, 60)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


Data: D:\Kuliah\DL\UPLOAD_GITHUB\Grokking-Deep-Learning\data
Mode cepat: True


In [2]:
with np.load(DATA_DIR / "mnist.npz") as mnist:
    images = mnist["x_train"][:(256 if MODE_CEPAT else 1000)].astype(float) / 255
    labels = mnist["y_train"][:len(images)]
    test_images = mnist["x_test"][:(200 if MODE_CEPAT else 1000)].astype(float) / 255
    test_labels = mnist["y_test"][:len(test_images)]

def patches(batch):
    view = np.lib.stride_tricks.sliding_window_view(batch, (3, 3), axis=(1, 2))
    return view.reshape(len(batch), -1, 9)

print("image -> patches:", images.shape, patches(images[:2]).shape)


image -> patches: (256, 28, 28) (2, 676, 9)


In [3]:
rng = np.random.default_rng(10)
num_kernels = 8
kernels = rng.normal(0, 0.05, (9, num_kernels))
feature_count = 26 * 26 * num_kernels
wout = rng.normal(0, 0.03, (feature_count, 10))
alpha, epochs, batch_size = 0.4, (3 if MODE_CEPAT else 12), 32
targets = np.eye(10)[labels]

for epoch in range(epochs):
    for start in range(0, len(images), batch_size):
        x = images[start:start+batch_size]
        y = targets[start:start+batch_size]
        p = patches(x)
        z = p.reshape(-1, 9) @ kernels
        hidden = np.tanh(z).reshape(len(x), -1)
        prob = softmax(hidden @ wout)
        dlogit = (prob - y) / len(x)
        dhidden = dlogit @ wout.T
        dz = dhidden.reshape(-1, num_kernels) * (1 - np.tanh(z) ** 2)
        wout -= alpha * hidden.T @ dlogit
        kernels -= alpha * p.reshape(-1, 9).T @ dz

def conv_predict(batch):
    p = patches(batch)
    hidden = np.tanh(p.reshape(-1, 9) @ kernels).reshape(len(batch), -1)
    return np.argmax(hidden @ wout, axis=1)

print("test accuracy:", np.mean(conv_predict(test_images) == test_labels))


test accuracy: 0.61


## Interpretasi dan kesimpulan

Hasil harus dibaca bersama asumsi, konfigurasi, seed, serta data yang digunakan. Keberhasilan menjalankan kode tidak membuktikan generalisasi, kausalitas, keamanan, atau kelayakan deployment. Bandingkan hasil dengan teori di atas dan jelaskan setiap perbedaan sebelum mengubah konfigurasi.
